# Compare training runs in a grid: 30 hyperparameter runs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toclocoinc/lattice-grid-notebooks/blob/main/experiment-comparison/experiment_comparison.ipynb)

One row per training run: its **hyperparameters**, its **metrics** with a colour scale, and its **loss curve as a sparkline** drawn per row on its own scale. The **best run is pinned** to the top, a dropdown **filters by hyperparameter**, and the runs you tick come back as a pandas `DataFrame`. No API key.

**Data:** the notebook runs the sweep itself: 30 small scikit-learn neural networks (`MLPClassifier`) with randomly drawn hyperparameters, trained on the [Optical Recognition of Handwritten Digits](https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits) data (Alpaydin & Kaynak, UCI, licensed **CC BY 4.0**; shipped inside scikit-learn as `load_digits`, so nothing to download). It takes about 10 seconds. The notebook code is MIT-licensed (see the repo `LICENSE`).

In [1]:
%pip install -q lattice-grid-jupyter

Note: you may need to restart the kernel to use updated packages.


## 1. Run the sweep

Each run draws a hidden-layer size, a learning rate, an L2 penalty and an activation from a seeded generator, trains for 20 epochs, and keeps the **validation accuracy**, the final **training loss** and the **loss curve** (the loss after each epoch from the second on, since the first epoch's loss dwarfs the rest; stored as a list in one cell).

In [2]:
import time, warnings
import numpy as np, pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

warnings.simplefilter("ignore")
X, y = load_digits(return_X_y=True)
Xtr, Xva, ytr, yva = train_test_split(StandardScaler().fit_transform(X), y, test_size=0.3, random_state=0)

rng = np.random.default_rng(7)
t0 = time.time()
runs = []
for i in range(30):
    hp = dict(hidden=int(rng.choice([16, 32, 64, 128])), lr=float(rng.choice([0.0003, 0.001, 0.003, 0.01, 0.03])),
              alpha=float(rng.choice([1e-5, 1e-3, 1e-1])), activation=str(rng.choice(["relu", "tanh"])))
    t = time.time()
    m = MLPClassifier((hp["hidden"],), activation=hp["activation"], learning_rate_init=hp["lr"], alpha=hp["alpha"],
                      max_iter=20, tol=0, n_iter_no_change=100, random_state=i).fit(Xtr, ytr)
    runs.append({"run": f"run-{i + 1:02d}", **hp, "val_acc": m.score(Xva, yva), "train_loss": m.loss_,
                 "seconds": time.time() - t, "loss_curve": [round(v, 4) for v in m.loss_curve_[1:]]})
sweep = pd.DataFrame(runs)
print(f"{len(sweep)} runs in {time.time() - t0:.0f}s; best val_acc {sweep.val_acc.max():.3f}")

30 runs in 5s; best val_acc 0.980


## 2. The comparison grid

* `loss_curve` is rendered by the grid's built-in **`line` cell renderer**: each row is scaled to its own minimum and maximum, so a slow run and a fast run both show their shape.
* `formatting` puts a **colour scale** on `val_acc` (white to green) and on `train_loss` (white to red).
* `pinned_top_rows` holds the **best run** (highest validation accuracy) above the scrolling rows.
* The **Hidden units** dropdown is a grid filter on `hidden`; the checkbox column selects runs.

In [3]:
from ipywidgets import Dropdown, VBox
from lattice_grid_jupyter import LatticeGridWidget

best = sweep.loc[[sweep.val_acc.idxmax()]].to_dict("records")  # the best run, pinned above the rest
green = {"scale": {"min": 0.85, "max": 1, "colours": ["#ffffff", "#2e9e5b"]}}
red = {"scale": {"min": 0, "max": 0.6, "colours": ["#ffffff", "#d64545"]}}
widths = dict(run=70, hidden=70, lr=70, alpha=75, activation=85, val_acc=85, train_loss=90, seconds=75)
grid = LatticeGridWidget(sweep, height=480, state={"sort": [{"col": "val_acc", "dir": "desc"}]}, options={
        "selection": {"mode": "multiple", "checkbox": True}, "pinned_top_rows": best,
        "formatting": {"val_acc": [green], "train_loss": [red]}},
    columns=[{"field": f, "layout": {"width": w}, **({"format": {"decimals": 3 if f != "seconds" else 2}} if f in ("val_acc", "train_loss", "seconds") else {})}
             for f, w in widths.items()]
    + [{"field": "loss_curve", "title": "loss per epoch", "cell": {"render": "line", "props": {"label": False}}, "layout": {"width": 180}}])

pick = Dropdown(options=["All runs"] + [f"hidden = {h}" for h in sorted(sweep.hidden.unique())], description="Hidden units")
def show(change):  # a hyperparameter filter is just grid state
    col, _, val = change.new.partition(" = ")
    grid.state = {**grid.state, "filters": {"op": "and", "conditions": [{"col": col, "op": "eq", "value": int(val)}] if val else []}}
pick.observe(show, "value")
VBox([pick, grid])

![the experiment grid](https://raw.githubusercontent.com/toclocoinc/lattice-grid-notebooks/main/experiment-comparison/grid.png)

## 3. Selected runs back as a DataFrame

`grid.selected` is the ticked runs as a pandas `DataFrame`, with the original index and dtypes (an empty frame when nothing is ticked). Tick a few runs above and run this cell again.

In [5]:
picked = grid.selected.drop(columns="loss_curve")
print(f"{len(picked)} runs selected")
picked.sort_values("val_acc", ascending=False)

3 runs selected


,run,hidden,lr,alpha,activation,val_acc,train_loss,seconds
17,run-18,128,0.01,0.10000,relu,0.977778,0.044538,0.211393
0,run-01,128,0.01,0.10000,tanh,0.974074,0.047515,0.255541
22,run-23,64,0.01,0.00001,tanh,0.972222,0.009038,0.165565
